In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
COLS_REQUERIDAS = [
    "segmento", "nit", "anio", "modalidad", "razon_social", "codigo_ciiu_4dig",
    "esf_Assets", "esf_Liabilities", "esf_Equity",
    "esf_CurrentAssets", "esf_CurrentLiabilities",
    "eri_Revenue", "eri_GrossProfit", "eri_ProfitLoss",
    "eri_GananciaPerdidaPorActividadesDeOperacion"
]
DATA_PATH = "../data/raw/panel_pymes.parquet"
df = pd.read_parquet(DATA_PATH, columns=COLS_REQUERIDAS)

In [41]:
# tipos de datos
df["nit"] = df["nit"].astype("string")

# filtro por tipos contables y divison entre 0
condiciones = (
    (df["anio"] >= 2017) & 
    (df["esf_Assets"] > 0) & 
    (df["esf_CurrentLiabilities"] > 0) & 
    (df["eri_Revenue"] > 0)
)

df = df[condiciones].copy()


print(df.shape)

(191344, 15)


In [42]:
# creacion de target supervisado
df["target"] = (df["eri_ProfitLoss"] < 0).astype(int)

target_anual = df.groupby("anio")["target"].agg(["count", "mean"]).reset_index()
target_anual.columns = ["Año", "Total Empresas", "Tasa Estrés"]
target_anual["Tasa Estrés (%)"] = (target_anual["Tasa Estrés"] * 100).round(2)
print(target_anual[["Año", "Total Empresas", "Tasa Estrés (%)"]])

    Año  Total Empresas  Tasa Estrés (%)
0  2017           15289            21.73
1  2018           14590            22.18
2  2019           19971            18.87
3  2020           23362            24.07
4  2021           22851            17.92
5  2022           24491            17.53
6  2023           24641            19.22
7  2024           23103            19.22
8  2025           23046            18.94


In [43]:
# Construccion de indicadores

# Patrimonio técnico negativo
df["patrimonio_negativo"] = (df["esf_Equity"] <= 0).astype(int)

# Margen Operacional acotado [-1.0, 1.0]
df["margen_operacional"] = np.where(
    df["eri_Revenue"] > 1000,
    df["eri_GananciaPerdidaPorActividadesDeOperacion"] / df["eri_Revenue"],
    0.0
).clip(-1.0, 1.0)

# Rotación de Activos (Ventas / Activos) acotado [0.0, 10.0]
df["rotacion_activos"] = np.where(
    df["esf_Assets"] > 0,
    df["eri_Revenue"] / df["esf_Assets"],
    0.0
).clip(0.0, 10.0)

# Endeudamiento (Pasivo / Activo) acotado [0.0, 5.0]
df["endeudamiento"] = np.where(
    df["esf_Assets"] > 0,
    df["esf_Liabilities"] / df["esf_Assets"],
    0.0
).clip(0.0, 5.0)

# Liquidez Estructural (Capital de Trabajo Neto / Activos) acotado [-2.0, 1.0]
df["liquidez_estructural"] = np.where(
    df["esf_Assets"] > 0,
    (df["esf_CurrentAssets"] - df["esf_CurrentLiabilities"]) / df["esf_Assets"],
    0.0
).clip(-2.0, 1.0)

ratios = ["patrimonio_negativo", "margen_operacional", "rotacion_activos", "endeudamiento", "liquidez_estructural"]

print(df[ratios].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

       patrimonio_negativo  margen_operacional  rotacion_activos  \
count        191344.000000       191344.000000      1.913440e+05   
mean              0.034629            0.107519      1.089296e+00   
std               0.182838            0.362848      1.426770e+00   
min               0.000000           -1.000000      5.196487e-09   
1%                0.000000           -1.000000      1.820320e-03   
5%                0.000000           -0.604198      1.502436e-02   
50%               0.000000            0.078894      6.754950e-01   
95%               0.000000            0.907400      3.628080e+00   
99%               1.000000            1.000000      7.538674e+00   
max               1.000000            1.000000      1.000000e+01   

       endeudamiento  liquidez_estructural  
count   1.913440e+05         191344.000000  
mean    4.933140e-01              0.256770  
std     3.918481e-01              0.333505  
min     1.507460e-07             -2.000000  
1%      4.198693e-03      

In [45]:
# Selección de columnas para modelado (X + target)
cols_export = [
    "patrimonio_negativo",
    "margen_operacional",
    "rotacion_activos",
    "endeudamiento",
    "liquidez_estructural",
    "target"
]

df_train = df[(df["anio"] >= 2017) & (df["anio"] <= 2022)][cols_export].copy()
df_test = df[(df["anio"] >= 2023) & (df["anio"] <= 2025)][cols_export].copy()

In [46]:
import os

OUTPUT_DIR = "../data/processed"

train_path = os.path.join(OUTPUT_DIR, "train_pymes_2017_2022.csv")
test_path = os.path.join(OUTPUT_DIR, "test_pymes_2023_2025.csv")

df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)